# **Pf8 Data Access**

<div class="alert alert-block alert-info">
<b>Note:</b> We are currently reviewing our compute and storage infrastructure. As part of this process, access to Pf8 data on S3 is being gradually retired. Pf8 analysis-ready data will remain accessible via Google Cloud and the <code>malariagen_data</code> Python package. Pf8 data files beyond the variants callset are also available via <a href="https://zenodo.org/records/18681980">Zenodo</a>.

The snp-only callset will remain available on S3 for now.
</div>

For a quick overview on the spatial and geographical distribution of the samples available, please visit our [Pf8 web-app](https://apps.malariagen.net/apps/pf8/).

## Google Cloud Access: how to authenticate
Pf8 data are accessible via Google Cloud Storage (GCS). Access to Google Cloud requires users to register for authentication. For instructions on how to authenticate, please see [here](https://malariagen.github.io/parasite-data/cloud_data_access/parasite-cloud-data-access.html).

## Acessing Pf8 with the malariagen_data Python package
This notebook illustrates how to read data directly from GCS, **without having to first download any data locally**. This notebook can be run from any computer, but will work best when run from a compute node within Google Cloud, because it will be physically closer to the data and so data transfer is faster. For example, this notebook can be run via MyBinder or Google Colab which are free interactive computing service running in the cloud.

To launch this notebook in the cloud and run it for yourself, click the launch icon (shaped like a rocket) at the top of the page and select one of the cloud computing services available.



## Setup

Running this notebook requires some Python packages to be installed. These packages can be installed via pip or conda. E.g.:

In [2]:
!pip install -q --no-warn-conflicts malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 40.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.2/265.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/2

To make accessing these data more convenient, we’ve created the malariagen_data Python package, which is available from PyPI. This is experimental so please let us know if you find any bugs or have any suggestions.

Now import the packages we’ll need to use here.

In [14]:
!apt-get update && apt-get install -y tabix


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,129 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,313 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,396 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packag

In [15]:
!bgzip --version
!tabix --version


bgzip (htslib) 1.13+ds
Copyright (C) 2021 Genome Research Ltd.
tabix (htslib) 1.13+ds
Copyright (C) 2021 Genome Research Ltd.


In [3]:
import numpy as np
import dask
import dask.array as da
from dask.diagnostics.progress import ProgressBar
import allel
# silence some warnings
dask.config.set(**{'array.slicing.split_large_chunks': False})
import malariagen_data

To access the pf8 data, use the following code:

In [4]:
pf8 = malariagen_data.Pf8()

## Metadata

Data on the samples that were sequenced as part of this resource are available. It includes the time and place of collection, quality metrics, and accession numbers.

To see all the information available, load sample metadata into a pandas dataframe:

In [5]:
pf8_metadata = pf8.sample_metadata()

#pf8_metadata.head()

In [5]:
print("The data set has {} samples and {} fields".format(pf8_metadata.shape[0],pf8_metadata.shape[1]))

The data set has 33325 samples and 17 fields


We can explore each of the fields:

- The <font color='purple'>Sample</font> column gives the unique sample identifier used throughout all Pf8 analyses.


- The <font color='purple'>Study</font> refers to the partner study which collected the sample.


- The <font color='purple'>Country</font> & <font color='purple'>Admin level 1</font> describe the location where the sample was collected from.


- The <font color='purple'>Country latitude</font>, <font color='purple'>Country longitude</font>, <font color='purple'>Admin level 1 latitude</font> and <font color='purple'>Admin 1 longitude</font> contain the GADM coordinates for each country & administrative level 1.


- The <font color='purple'>Year</font> column gives the time of sample collection.


- The <font color='purple'>ENA</font> column gives the run accession(s) for the sequencing read data for each sample.


- The <font color='purple'>All samples same case</font> column identifies samples set collected from the same individual.


- The <font color='purple'>Population</font> column gives the population to which the sample has been assigned. The possible values are: Africa - West (AF-W), Africa-Central (AF-C), Africa - East (AF-E), Africa - Northeast (AF-NE), Asia - South - East (AS-S-E), Asia - South – Far East (AS-S-FE), Asia - Southeast - West (AS-SE-W), Asia - Southeast - East (AS-SE-E), Oceania - New Guinea (OC-NG), South America (SA).


- The <font color='purple'>% callable</font> column refers to the %  of the genome with coverage of at least 5 reads and less than 10% of reads with mapping quality 0.


- The <font color='purple'>QC pass</font> column defines whether the sample passed (True) or failed (False) QC.
    
    
- The <font color='purple'>Exclusion reason</font> describes the reason why the particular sample was excluded from the main analysis.
    
    
- The <font color='purple'>Sample type</font> column gives details on the DNA preparation method used
    
    
- The <font color='purple'>Sample was in Pf7</font> column defines whether the sample was included in the previous version of the data release (Pf7) or if it is new to Pf8.

The python package [Pandas](https://pandas.pydata.org/) can be used to explore and query the sample metadata in different ways. For example, here is a summary of the numbers of samples grouped by the country they were collected in:

In [7]:
pf8_metadata.groupby("Country").size()

,0
Country,
Bangladesh,1658
Benin,334
Burkina Faso,58
Cambodia,2282
Cameroon,294
Colombia,167
Côte d'Ivoire,71
Democratic Republic of the Congo,1549
Ethiopia,35


In [8]:
# Extract Gambian and Senegalese sample IDs
gmbsen_ids = pf8_metadata.query("Country == 'Gambia' | Country == 'Senegal'")

In [ ]:
# Get list of samples IDs

sample_ids = gmbsen_ids.index.values
print(sample_ids)

In [ ]:
print(f"Found {len(sample_ids)} samples for both countries")

In [ ]:
gmbsen_ids.values

In [9]:
# Filter only sample that pass the QC
gmbsen_ids_pass = gmbsen_ids[gmbsen_ids['QC pass'] == True]

In [ ]:
print(gmbsen_ids_pass)

## Variant Calls

Two variant callset versions were created for Pf8 from all samples in the release:
- a **Full callset:** Contains details of 12,493,205 discovered variant genome positions. Variants are single nucleotide polymorphisms (SNPs), short insertion/deletions (indels), or a combination of SNPs and indels.
- a **SNP-only callset:** A subset of the full callset, with all indel variants removed, leaving only SNPs. There are 10,821,552 SNPs in this callset.

Data on variant calls, including the genomic positions, alleles, and genotypes, can be accessed as an xarray Dataset:

In [10]:
# Access the full callset
variant_dataset = pf8.variant_calls()

In [11]:
# Access the SNP-only callset
pf8_snp_only  = malariagen_data.Pf8('https://pf8-release.cog.sanger.ac.uk/snp-only')
variant_dataset_snp_only = pf8_snp_only.variant_calls()

In [12]:
# Sample IDs in genomic data
all_ids = variant_dataset_snp_only["sample_id"].values

In [ ]:
# 1. Subset for Gambia/Senegal (QC passed) and limit to Chromosome 7 for stability
sample_mask = np.isin(variant_dataset_snp_only['sample_id'], gmbsen_ids['Sample'])

# FIX: Added .compute() to the boolean mask to avoid KeyError with Dask arrays
chrom_mask = (variant_dataset_snp_only.variant_chrom == 'Pf3D7_01_v3').compute()
variant_dataset_region = variant_dataset_snp_only.isel(variants=chrom_mask)
variant_dataset_subset = variant_dataset_region.isel(samples=sample_mask)

# 2. Access genotype data for this region
gt_region = variant_dataset_subset['call_genotype'].data
n_samples = variant_dataset_subset.dims['samples']

print(f"Calculating missingness for {n_samples} samples on Chromosome 7...")
with ProgressBar():
    # Calculate missingness fraction (where genotype is -1)
    variant_missing_frac = (gt_region == -1).any(axis=2).mean(axis=1).compute()

# 3. Define filters
mask_low_missing = variant_missing_frac <= 0.50
mask_biallelic = (variant_dataset_subset['variant_numalt'].data == 1).compute()

# Combined mask
variant_mask = mask_low_missing & mask_biallelic

# 4. Apply filter to the dataset
variant_dataset_filtered = variant_dataset_subset.isel(variants=variant_mask)

print(f"Original variants on Chrom 7: {len(variant_dataset_subset.variants)}")
print(f"Filtered variants (missingness <= 50% & biallelic): {len(variant_dataset_filtered.variants)}")
display(variant_dataset_filtered)

### Genome-wide Filtering by Chromosome
To avoid `ServerDisconnectedError` and ensure data persistence, we will iterate through all contigs, filter variants, and collect metrics to evaluate sample quality.

In [13]:
import os
import pandas as pd
import numpy as np
import time
import allel
import dask
from dask.diagnostics.progress import ProgressBar

# 1. Setup sample subsetting - Ensuring indices are computed once
sample_indices = np.where(np.isin(variant_dataset_snp_only['sample_id'], gmbsen_ids['Sample']))[0]
sample_ids_list = list(variant_dataset_snp_only['sample_id'].values[sample_indices])

contigs = pf8.contigs
total_variants_filtered = 0
sample_missing_counts = np.zeros(len(sample_indices), dtype=int)

print(f"Iterating through chromosomes for {len(sample_indices)} samples...")

for chrom in contigs:
    progress_file = f"stats_{chrom}.npy"
    vcf_output = f"filtered_{chrom}.vcf"

    if os.path.exists(progress_file):
        stats = np.load(progress_file, allow_pickle=True).item()
        total_variants_filtered += stats['count']
        sample_missing_counts += stats['s_missing']
        print(f"--- {chrom}: Resumed from cache ({stats['count']} variants) ---")
        continue

    print(f"--- {chrom}: Processing (Ultra-stable 2k chunks) ---")
    chrom_mask = (variant_dataset_snp_only.variant_chrom == chrom).compute()
    ds_chrom = variant_dataset_snp_only.isel(variants=chrom_mask)

    n_variants = ds_chrom.sizes['variants']
    if n_variants == 0:
        continue

    chunk_size = 10000
    chrom_total_filtered = 0
    chrom_sample_missing = np.zeros(len(sample_indices), dtype=int)

    # Lists to store filtered data for VCF export
    v_chroms, v_pos, v_ref, v_alt, v_gt = [], [], [], [], []

    for start in range(0, n_variants, chunk_size):
        end = min(start + chunk_size, n_variants)
        ds_chunk = ds_chrom.isel(variants=slice(start, end))

        # Filter biallelic sites
        is_biallelic = (ds_chunk['variant_numalt'].data == 1).compute()

        if np.any(is_biallelic):
            ds_biallelic = ds_chunk.isel(variants=is_biallelic, samples=sample_indices)
            gt_biallelic = ds_biallelic['call_genotype'].data

            max_retries = 7
            for attempt in range(max_retries):
                try:
                    v_miss = (gt_biallelic == -1).any(axis=2).mean(axis=1).compute()
                    v_mask = (v_miss <= 0.50)
                    num_f = np.sum(v_mask)

                    if num_f > 0:
                        gt_passed = gt_biallelic[v_mask].compute()
                        s_miss = (gt_passed == -1).any(axis=2).sum(axis=0)
                        chrom_total_filtered += num_f
                        chrom_sample_missing += s_miss

                        ds_passed = ds_biallelic.isel(variants=v_mask)
                        # Explicitly compute metadata to avoid dask object issues in concatenation
                        v_chroms.append(ds_passed['variant_chrom'].values)
                        v_pos.append(ds_passed['variant_position'].values)
                        v_ref.append(ds_passed['variant_allele'].values[:, 0])
                        v_alt.append(ds_passed['variant_allele'].values[:, 1])
                        v_gt.append(gt_passed)
                    break
                except Exception as e:
                    if attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 20
                        print(f"Retrying {chrom} chunk {start} (attempt {attempt+1}) due to connection drop...")
                        time.sleep(wait_time)
                    else:
                        raise e

    if chrom_total_filtered > 0:
        # Prepare callset dictionary for write_vcf
        # Note: 'variants/ALT' must be 2D. 'calldata/GT' is recognized for genotypes.
        callset = {
            'variants/CHROM': np.concatenate(v_chroms),
            'variants/POS': np.concatenate(v_pos),
            'variants/REF': np.concatenate(v_ref),
            'variants/ALT': np.concatenate(v_alt)[:, np.newaxis],
            'calldata/GT': np.concatenate(v_gt),
            'samples': sample_ids_list
        }

        print(f"Writing {chrom_total_filtered} variants to {vcf_output}...")
        # We pass the samples separately to ensure column headers are created
        allel.write_vcf(vcf_output, callset, samples=sample_ids_list)

        total_variants_filtered += chrom_total_filtered
        sample_missing_counts += chrom_sample_missing
        np.save(progress_file, {'count': chrom_total_filtered, 's_missing': chrom_sample_missing})
        print(f"Completed {chrom}: Found {chrom_total_filtered} variants")

# Final results processing
if total_variants_filtered > 0:
    quality_df = pd.DataFrame({
        'Sample': sample_ids_list,
        'missing_fraction': sample_missing_counts / total_variants_filtered
    })
    quality_df.to_csv('sample_global_missingness.csv', index=False)
    print("\nGlobal Processing Complete.")
    display(quality_df.head())
else:
    print("No variants passed filters.")

Iterating through chromosomes for 2153 samples...
--- Pf3D7_01_v3: Processing (Ultra-stable 2k chunks) ---
Writing 173892 variants to filtered_Pf3D7_01_v3.vcf...
Completed Pf3D7_01_v3: Found 173892 variants
--- Pf3D7_02_v3: Processing (Ultra-stable 2k chunks) ---


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_449/781048289.py", line 72, in <cell line: 0>
    v_ref.append(ds_passed['variant_allele'].values[:, 0])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xarray/core/dataarray.py", line 798, in values
    return self.variable.values
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xarray/core/variable.py", line 556, in values
    return _as_array_or_item(self._data)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xarray/core/variable.py", line 336, in _as_array_or_item
    data = np.asarray(data)
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/dask/array/core.py", line 1720, in __array__
    x = self.compute()
        ^^^^^

TypeError: object of type 'NoneType' has no len()

In [ ]:
import os
import json
import hashlib
import time
import numpy as np
import dask
from dask.diagnostics.progress import ProgressBar
import pandas as pd

# ------------------------------------------------------------------
# Config -- both values feed the cache hash below, so changing either
# and rerunning will correctly force reprocessing rather than reusing
# stale cached chromosomes.
# ------------------------------------------------------------------
MISSING_THRESHOLD = 0.50
CHUNK_SIZE = 10000


def compute_config_hash(missing_threshold, chunk_size, sample_ids):
    cfg = {
        'missing_threshold': missing_threshold,
        'chunk_size': chunk_size,
        'sample_ids': sorted(sample_ids),
    }
    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]


# ------------------------------------------------------------------
# Contig lengths for the VCF header
# ------------------------------------------------------------------
def get_contig_lengths(pf8, contigs):
    """
    Pulls real contig lengths from the reference genome attached to pf8.
    This assumes a malariagen_data-style API where pf8.genome_sequence(contig)
    returns the reference array and its length IS the contig length.
    VERIFY this against your actual pf8 object -- if it exposes lengths via
    something else (e.g. genome_features, a contig_sizes dict, a FASTA
    index), swap the body of this function. Falls back to omitting length
    from the header rather than guessing from max(POS), since that is not
    the true contig length.
    """
    lengths = {}
    for contig in contigs:
        try:
            lengths[contig] = int(pf8.genome_sequence(region=contig).shape[0])
        except Exception as e:
            print(f"Warning: could not determine length for {contig} ({e}); "
                  f"##contig header will omit length for this contig.")
            lengths[contig] = None
    return lengths

"""
def get_contig_lengths(pf8, contigs):
    lengths = {}
    for contig in contigs:
        try:
            lengths[contig] = int(pf8.genome_sequence(region=contig).shape[0])
        except Exception as e:
            print(f"Warning: could not determine length for {contig} ({e}); "
                  f"##contig header will omit length for this contig.")
            lengths[contig] = None
    return lengths
"""

def write_vcf_header(fh, contigs, contig_lengths, filter_ids, sample_ids_list):
    fh.write("##fileformat=VCFv4.2\n")
    fh.write(f"##fileDate={time.strftime('%Y%m%d')}\n")
    fh.write("##source=custom_gmbsen_subsetting_pipeline\n")
    for contig in contigs:
        length = contig_lengths.get(contig)
        if length:
            fh.write(f"##contig=<ID={contig},length={length}>\n")
        else:
            fh.write(f"##contig=<ID={contig}>\n")
    fh.write('##FILTER=<ID=PASS,Description="All filters passed">\n')
    for fid in sorted(filter_ids - {'PASS'}):
        # Generic description -- replace with the real filter definition
        # from your source dataset's documentation if you have it.
        fh.write(f'##FILTER=<ID={fid},Description="Filter flag carried over from source dataset">\n')
    fh.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
    header_cols = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT'] + list(sample_ids_list)
    fh.write('\t'.join(header_cols) + '\n')


# ------------------------------------------------------------------
# Genotype formatting + body writing
# ------------------------------------------------------------------
def genotypes_to_vcf_strings(gt_array):
    """
    gt_array: (n_variants, n_samples, ploidy) int array, -1 = missing.
    Returns (n_variants, n_samples) array of unphased GT strings, e.g. '0/1', './.'.
    """
    ploidy = gt_array.shape[2]
    allele_strs = np.where(gt_array == -1, '.', gt_array.astype(str))
    gt_strings = allele_strs[..., 0]
    for p in range(1, ploidy):
        gt_strings = np.char.add(np.char.add(gt_strings, '/'), allele_strs[..., p])
    return gt_strings


def write_vcf_body_chunk(fh, chrom, pos, ref, alt, filter_col, gt_strings):
    n_variants, n_samples = gt_strings.shape
    # NOTE: if ref/alt come through as numpy bytes dtype ('S'), .astype(str)
    # decodes correctly. If they're Python `bytes` objects inside an object
    # array instead, .astype(str) will give literal "b'A'" strings -- check
    # ds_passed['variant_allele'].values.dtype once and add .astype('U')
    # or a .decode() step if needed.
    fixed_cols = np.column_stack([
        chrom.astype(str),
        pos.astype(str),
        np.full(n_variants, '.'),   # ID
        ref.astype(str),
        alt.astype(str),
        np.full(n_variants, '.'),   # QUAL
        filter_col,
        np.full(n_variants, '.'),   # INFO
        np.full(n_variants, 'GT'),  # FORMAT
    ])
    full = np.column_stack([fixed_cols, gt_strings])
    fh.write('\n'.join('\t'.join(row) for row in full) + '\n')


# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------
sample_indices = np.where(np.isin(variant_dataset_snp_only['sample_id'], gmbsen_ids['Sample']))[0]
sample_ids_list = list(variant_dataset_snp_only['sample_id'].values[sample_indices])

contigs = pf8.contigs
contig_lengths = get_contig_lengths(pf8, contigs)
config_hash = compute_config_hash(MISSING_THRESHOLD, CHUNK_SIZE, sample_ids_list)

# Boolean pass/fail is the only FILTER info this script currently carries,
# so the header's possible values are fixed up front -- no need to discover
# them mid-stream, which would otherwise conflict with writing the header
# before the body.
has_filter_field = 'variant_filter_pass' in variant_dataset_snp_only
filter_ids_for_header = {'PASS', 'FAIL'} if has_filter_field else {'PASS'}

total_variants_filtered = 0
sample_missing_counts = np.zeros(len(sample_indices), dtype=int)

print(f"Iterating through chromosomes for {len(sample_indices)} samples...")

for chrom in contigs:
    progress_file = f"stats_{chrom}.npy"
    vcf_output = f"filtered_{chrom}.vcf"

    if os.path.exists(progress_file) and os.path.exists(vcf_output):
        stats = np.load(progress_file, allow_pickle=True).item()
        if stats.get('config_hash') == config_hash:
            total_variants_filtered += stats['count']
            sample_missing_counts += stats['s_missing']
            print(f"--- {chrom}: Resumed from cache ({stats['count']} variants) ---")
            continue
        else:
            print(f"--- {chrom}: Cache config changed (threshold/chunk_size/sample list) -- reprocessing ---")
    elif os.path.exists(progress_file) and not os.path.exists(vcf_output):
        print(f"--- {chrom}: Stats cache found but VCF file missing -- reprocessing ---")

    print(f"--- {chrom}: Processing (chunk size {CHUNK_SIZE}) ---")
    chrom_mask = (variant_dataset_snp_only.variant_chrom == chrom).compute()
    ds_chrom = variant_dataset_snp_only.isel(variants=chrom_mask)

    n_variants = ds_chrom.sizes['variants']
    if n_variants == 0:
        continue

    chrom_total_filtered = 0
    chrom_sample_missing = np.zeros(len(sample_indices), dtype=int)

    with open(vcf_output, 'w') as vcf_fh:
        write_vcf_header(vcf_fh, contigs, contig_lengths, filter_ids_for_header, sample_ids_list)

        for start in range(0, n_variants, CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, n_variants)
            ds_chunk = ds_chrom.isel(variants=slice(start, end))

            is_biallelic = (ds_chunk['variant_numalt'].data == 1).compute()
            if not np.any(is_biallelic):
                continue

            ds_biallelic = ds_chunk.isel(variants=is_biallelic, samples=sample_indices)
            gt_biallelic = ds_biallelic['call_genotype'].data

            max_retries = 7
            for attempt in range(max_retries):
                try:
                    v_miss = (gt_biallelic == -1).any(axis=2).mean(axis=1).compute()
                    v_mask = (v_miss <= MISSING_THRESHOLD)
                    num_f = int(np.sum(v_mask))

                    if num_f == 0:
                        break

                    # --- everything below is staged into LOCAL variables.
                    # Nothing mutates chrom-level state or hits disk until
                    # the whole chunk has computed successfully -- so a
                    # mid-chunk failure on retry can't leave partial /
                    # misaligned data behind.
                    gt_passed = gt_biallelic[v_mask].compute()
                    s_miss = (gt_passed == -1).any(axis=2).sum(axis=0)

                    ds_passed = ds_biallelic.isel(variants=v_mask)
                    chunk_chrom = ds_passed['variant_chrom'].values
                    chunk_pos = ds_passed['variant_position'].values
                    chunk_ref = ds_passed['variant_allele'].values[:, 0]
                    chunk_alt = ds_passed['variant_allele'].values[:, 1]

                    if has_filter_field:
                        chunk_filter = np.where(
                            ds_passed['variant_filter_pass'].values, 'PASS', 'FAIL'
                        )
                    else:
                        chunk_filter = np.full(num_f, 'PASS')

                    chunk_gt_strings = genotypes_to_vcf_strings(gt_passed)

                    # all computes succeeded -- safe to commit now
                    chrom_total_filtered += num_f
                    chrom_sample_missing += s_miss
                    write_vcf_body_chunk(
                        vcf_fh, chunk_chrom, chunk_pos, chunk_ref, chunk_alt,
                        chunk_filter, chunk_gt_strings
                    )
                    break
                except Exception as e:
                    if attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 20
                        print(f"Retrying {chrom} chunk {start} (attempt {attempt+1}) due to: {e}")
                        time.sleep(wait_time)
                    else:
                        raise

    total_variants_filtered += chrom_total_filtered
    sample_missing_counts += chrom_sample_missing
    np.save(progress_file, {
        'count': chrom_total_filtered,
        's_missing': chrom_sample_missing,
        'config_hash': config_hash,
    })
    print(f"Completed {chrom}: {chrom_total_filtered} variants written"
          + ("" if chrom_total_filtered > 0 else " (empty VCF, header only)"))

# ------------------------------------------------------------------
# Final results
# ------------------------------------------------------------------
if total_variants_filtered > 0:
    import pandas as pd
    quality_df = pd.DataFrame({
        'Sample': sample_ids_list,
        'missing_fraction': sample_missing_counts / total_variants_filtered
    })
    quality_df.to_csv('sample_global_missingness.csv', index=False)
    print("\nGlobal Processing Complete.")
    display(quality_df.head())
else:
    print("No variants passed filters.")

import subprocess
for chrom in contigs:
    vcf_path = f"filtered_{chrom}.vcf"
    if os.path.exists(vcf_path):
        subprocess.run(['bgzip', '-f', vcf_path], check=True)
        subprocess.run(['tabix', '-p', 'vcf', f"{vcf_path}.gz"], check=True)

Iterating through chromosomes for 2153 samples...
--- Pf3D7_01_v3: Processing (chunk size 10000) ---
Completed Pf3D7_01_v3: 173892 variants written
--- Pf3D7_02_v3: Processing (chunk size 10000) ---
Completed Pf3D7_02_v3: 265811 variants written
--- Pf3D7_03_v3: Processing (chunk size 10000) ---
Completed Pf3D7_03_v3: 307084 variants written
--- Pf3D7_04_v3: Processing (chunk size 10000) ---
Completed Pf3D7_04_v3: 326050 variants written
--- Pf3D7_05_v3: Processing (chunk size 10000) ---
Completed Pf3D7_05_v3: 410709 variants written
--- Pf3D7_06_v3: Processing (chunk size 10000) ---
Completed Pf3D7_06_v3: 406565 variants written
--- Pf3D7_07_v3: Processing (chunk size 10000) ---


In [ ]:
import allel

# Prepare the data for VCF export
# Extract fixed fields
chroms = variant_dataset_filtered['variant_chrom'].values
pos = variant_dataset_filtered['variant_position'].values
ref = variant_dataset_filtered['variant_allele'].values[:, 0]
# variant_allele[:, 1] is the first alternate allele (since we filtered for biallelic)
alt = variant_dataset_filtered['variant_allele'].values[:, 1]

# Extract genotypes
gt = variant_dataset_filtered['call_genotype'].values

# Define output filename
vcf_output = 'filtered_gmbsen_chrom7.vcf'

print(f"Writing {len(pos)} variants to {vcf_output}...")

# Use positional arguments for core fields to avoid parameter name conflicts in allel versions
allel.write_vcf(
    vcf_output,
    chroms,
    pos,
    ref,
    [alt],
    genotypes=gt,
    samples=variant_dataset_filtered['sample_id'].values
)

print("VCF export complete.")

### Visualize Global Sample Missingness
Now we can visualize which samples have high missingness across the high-quality biallelic sites we've identified.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'quality_df' in locals():
    plt.figure(figsize=(10, 6))
    sns.histplot(quality_df['missing_fraction'], bins=50, kde=True)
    plt.axvline(0.1, color='red', linestyle='--', label='10% Missing Threshold')
    plt.title('Global Missingness per Sample (Across Filtered SNPs)')
    plt.xlabel('Proportion of Missing Genotypes')
    plt.ylabel('Count of Samples')
    plt.legend()
    plt.show()

    # Identify high quality samples
    pass_samples = quality_df[quality_df['missing_fraction'] < 0.10]['Sample']
    print(f"Samples passing < 10% missingness: {len(pass_samples)} out of {len(quality_df)}")
else:
    print("Please run the chromosome processing cell above first.")

In [ ]:
# Extract the target sample ids from the VCFs
variant_dataset_snp_only = variant_dataset_snp_only.sel(samples=sample_ids)
variant_dataset_snp_only

### Filtering Variants by Missingness and Allelic State
We calculate the missingness per variant and filter the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# To avoid ServerDisconnectedError, we subset the variants (e.g., every 10th variant)
# this reduces the data transfer while maintaining a representative distribution.
gt_target_subset = variant_dataset_snp_only["call_genotype"].data[::10]

# Calculate missing values (where genotype is -1)
# We perform the sum and compute on the subsetted array
missing_per_sample = (gt_target_subset == -1).sum(axis=(0, 2)).compute()

# Plot the distribution
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(missing_per_sample, bins=50, kde=True, ax=ax)
ax.set_title('Distribution of Missing Genotype Data Per Sample (Subsampled Variants)')
ax.set_xlabel('Number of Missing Genotypes (in subset)')
ax.set_ylabel('Number of Samples')
plt.tight_layout()
plt.show()

The default returns a basic set of data most commonly used for data analysis. However, for more complex analysis the full range of variables available in the zarr can be accessed by setting the extended flag to `True`, as shown below:

## Genotypes

Genotypes for individual samples are available.

Genotypes are stored as a three-dimensional array, where:

* the first dimension corresponds to genomic positions,
* the second dimension is samples,
* the third dimension is ploidy (2).

Values coded as integers, where -1 represents a missing value, 0 represents the reference allele, and 1, 2, and 3 represent alternate alleles.

Variant genotypes can be accessed as dask arrays as shown below.

In [ ]:
gt = variant_dataset["call_genotype"].data
gt

Note that the columns of this array (second dimension) match the rows in the sample metadata. You can use this correspondance to apply further subsetting operations to the genotypes by querying the sample metadata. E.g.:

In [ ]:
loc_colombia = pf8_metadata.eval("Country == 'Colombia'").values
print(f"found {np.count_nonzero(loc_colombia)} samples from Colombia")
variant_dataset_colombia = variant_dataset.isel(samples=loc_colombia)
variant_dataset_colombia